# 3. Préparation ML — Analyse des données et table Gold

Notebook d'analyse (séparé du notebook Silver, qui reste le pipeline de production planifié) : à partir de la table Silver `workspace.projet_alpha.insurance_claim_fraud_silver`, il réalise :
1. **Analyse de la cible** : taux de fraude (20% / 80%)
2. **Chasse aux fuites de données** : corrélations numériques vs cible, taux de fraude par modalité des catégorielles
3. **Verdicts** : features exclues (post-décision, générateur, sans signal) — justifications documentées dans les cellules markdown
4. **Liste des features Gold** : la spécification de la table Gold construite à la suite

Tout ce qui est exécuté ici relit la table Silver depuis Unity Catalog : aucun effet de bord sur le pipeline ETL quotidien.

In [0]:
# Paramètres du notebook
silver_table = "workspace.projet_alpha.insurance_claim_fraud_silver"
table_silver = spark.table(silver_table)

from pyspark.sql import functions as F

In [0]:
# Colonne cible
target = table_silver.select("fraudulent")
display(target)

In [0]:
# Taux de fraude / non fraude dans la Silver
taux_fraude = spark.sql(f"""
    SELECT count(CASE WHEN fraudulent = 'YES' THEN 1 END) AS nb_fraudes,
    count(CASE WHEN fraudulent = 'NO' THEN 0 END) AS nb_non_fraudes,
           count(*)                                             AS nb_total,
           round(count(CASE WHEN fraudulent = 'YES' THEN 1 END) / count(*) * 100, 2) AS taux_fraude_pct,
           round(count(CASE WHEN fraudulent = 'NO' THEN 1 END) / count(*) * 100, 2) AS taux_non_fraude_pct
    FROM {silver_table}
""")
display(taux_fraude)

In [0]:
%pip install seaborn

In [0]:
# Heatmap de corrélation : variables numériques + cible encodée en 0/1
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cible "fraudulent" ('YES'/'NO') encodée en 0/1 dans une copie — la colonne d'origine reste inchangée
table_silver_num = table_silver.withColumn(
    "fraudulent_num",
    (F.col("fraudulent") == "YES").cast("int")   # 'YES' -> 1, tout le reste (dont 'NO') -> 0
)

numeric_cols = [f.name for f in table_silver.schema.fields if f.dataType.simpleString() in ("int", "double", "bigint")]
pdf_corr = table_silver_num.select(numeric_cols + ["fraudulent_num"]).toPandas().corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(pdf_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Matrice de corrélation — variables numériques + cible 0/1 (Silver)", fontsize=14)
plt.tight_layout()
plt.show()

# Corrélation de chaque variable numérique avec la cible, triée par magnitude
corr_target = (pdf_corr["fraudulent_num"]
               .drop("fraudulent_num")   # hors autocorrélation
               .sort_values(key=abs, ascending=False))
print("\nCorrélation avec la cible 'fraudulent' (0/1) :")
print(corr_target.to_string())

In [0]:
# Distribution de fraud_risk_score
table_silver = spark.sql(f"SELECT * FROM {silver_table}")
risk_score = table_silver.select("fraud_risk_score")
display(risk_score)
display(risk_score.summary())

### Verdict sur les features corrélés avec la cible

On choisit comme stratégie de conserver les features que le client peut rentrer lors de sa déclaration de sinistre plutôt que des features liées à l'enquête de l'assureur.

1) fraud_risk_score (0.81): exclu car c'est le score qui définit la cible
2) approved_amount (-0.33) & payout_amount (-0.29) : exclus car résultats de la décision de l'assureur, inconnus au moment de la déclaration

#### Features redondantes
- vehicle_age - vehicle_year : on garde vehicle_age (plus interprétable que vehicle_year), on écarte le doublon
- approved_amount - payout_amount : on ne garde pas approved_amount car c'est le montant approuvé par l'assureur après études du dossier, donc dépend d'une enquête
- claim_amount - repair_cost : repair_cost apporte des informations liées au véhicule accidenté donc on le garde car le client peut l'indiquer en regardant le montant de ses factures chez le garagiste. claim_amount est le montant réclamé par le sinistré donc on le garde car dans la heatmap, corr(claim_amount, payout_amount) = 0.77 (montant réclamé différent du montant versé).

#### Indicateurs de fraude

| Seau | Features | Pourquoi |
|------|----------|----------|
| Connu dès la déclaration (contrôles automatiques) | claim_submitted_at_night, delayed_reporting, multiple_claims_last_year, gps_location_match, suspicious_documents | Calculables automatiquement au moment du dépôt, sans intervention d'un enquêteur |
| Nécessite un processus/une vérification | identity_verified, repair_shop_verified, inconsistent_statements, claim_history_risk | Disponibles seulement après vérification, entretien ou analyse du dossier — donc après le début de l'enquête |
| Le générateur lui-même | fraud_risk_score | C'est le score qui définit la cible — exclure |

NB : multiple_claims_last_year, bien que connu dès la déclaration, sera finalement abandonné à l'étape du dépistage des catégorielles (signal trop faible).

In [0]:
# Dépistage des catégorielles (liste des fraud indicators + candidats)
colonnes = table_silver.columns

colonnes_categorielles = [f.name for f in table_silver.schema.fields if f.dataType.simpleString() in ("string") and f.name in("claim_submitted_at_night", "delayed_reporting", "multiple_claims_last_year", "gps_location_match", "suspicious_documents","identity_verified", "repair_shop_verified", "inconsistent_statements", "claim_history_risk", "incident_type", "accident_severity", "police_report", "total_loss", "claim_status")]
print(colonnes_categorielles)
print(len(colonnes_categorielles))

In [0]:
# Taux de fraude par modalité de chaque catégorielle
for colonne in colonnes_categorielles:
  print(f"taux de fraude par {colonne} (SQL)")
  print(spark.sql(f"""
      SELECT {colonne},
             COUNT(*)                                        AS nb_sinistres,
             SUM(CASE WHEN fraudulent = 'YES' THEN 1 ELSE 0 END) AS nb_fraudes,
             ROUND(AVG(CASE WHEN fraudulent = 'YES' THEN 1 ELSE 0 END) * 100, 2) AS taux_fraude_pct
      FROM {silver_table}
      GROUP BY {colonne}
      ORDER BY taux_fraude_pct DESC
  """).toPandas())
  print("----------------------------------------------------")

#### Analyse des features catégorielles en lien avec la cible

Le modèle prédit au moment du dépôt, à partir des données saisies par le sinistré.

**1) Modalités en lien direct avec la cible**
- suspicious_documents (YES) : taux de fraude (93%), calculable au dépôt du sinistre, capture 7553/20 042 fraudes, soit 38% de toutes les fraudes (signal fort pour le modèle)

**2) Ecart fort et symétrique => signal honnête, on garde**
- accident_severity : taux de fraude variable selon la catégorie
- police_report : taux de fraude avec écart (24% VS 18%)
- claim_submitted_at_night : taux de fraude avec écart (34% VS 18%)
- gps_location_match : taux de fraude avec écart (49% VS 16%), le contrôle GPS est calculé instantanément à partir des données de déclaration
- delayed_reporting : 21% d'écart

**3) Aucun écart => abandon**
- incident_type : tout à 20% de taux de fraude
- multiple_claims_last_year : 5% d'écart de taux de fraude

**4) Features nécessitant une enquête => abandon**
- claim_status : 55% taux de fraude avec DENIED et APPROVED : taux de fraude à 0% => statut résulte de la décision (on n'approuve pas les fraudeurs), 100% des fraudes sont dans le statut DENIED
- inconsistent_statements : taux de fraude avec écart (50% VS 17%)
- repair_shop_verified : 10% d'écart
- claim_history_risk : 40% puis 20% de taux de fraude
- identity_verified (NO) : taux de fraude (69%), échec de vérification d'identité = fraude. La vérification d'identité est un processus dont le résultat n'est pas encore disponible au moment où le modèle doit prédire.
- total_loss : 9% d'écart; on remarque toutefois qu'il s'agit d'une information incluse dans accident_severity donc on préfera l'exclure et conserver accident_severity pour éviter la duplication d'informations

In [0]:
# Dépistage des autres catégorielles (hors fraud indicators)
columns_cat_others = [f.name for f in table_silver.schema.fields if f.dataType.simpleString() in ("string") and f.name not in("claim_submitted_at_night", "delayed_reporting", "multiple_claims_last_year", "gps_location_match", "suspicious_documents","identity_verified", "repair_shop_verified", "inconsistent_statements", "claim_history_risk", "incident_type", "accident_severity", "police_report", "total_loss", "claim_status","policy_id", "claim_id","customer_id" , "customer_name","source_file", "fraudulent", "source_bronze_table", "silver_batch_id")]
print(columns_cat_others)
print(len(columns_cat_others))

for colonne in columns_cat_others:
  print(f"taux de fraude par {colonne} (SQL)")
  print(spark.sql(f"""
      SELECT {colonne},
             COUNT(*)                                        AS nb_sinistres,
             SUM(CASE WHEN fraudulent = 'YES' THEN 1 ELSE 0 END) AS nb_fraudes,
             ROUND(AVG(CASE WHEN fraudulent = 'YES' THEN 1 ELSE 0 END) * 100, 2) AS taux_fraude_pct
      FROM {silver_table}
      GROUP BY {colonne}
      ORDER BY taux_fraude_pct DESC
  """).toPandas())
  print("----------------------------------------------------")

Les autres features catégorielles hors de la catégorie Fraud Indicators peuvent être exclues car elles n'apportent pas de signaux forts sur le taux de fraude des sinistres (tous à ~20%).

#### Liste des features Gold

**Cible**
- `fraudulent` (encodée 0/1 dans la Gold : YES → 1, NO → 0)

**Numériques gardées (19)** — toutes connues au moment du dépôt
- Client : `age`, `credit_score`, `years_with_company`, `previous_claims`, `annual_income`
- Police : `policy_duration_years`, `premium_amount`, `coverage_amount`, `deductible`
- Incident : `witnesses`, `injuries`
- Véhicule : `vehicle_age`, `mileage`, `vehicle_value`
- Coûts déclarés par le sinistré : `repair_cost`, `medical_cost`, `towing_cost`, `rental_vehicle_cost`
- Réclamation : `claim_amount` — montant réclamé par le sinistré (T0), confirmé par corr(claim_amount, payout_amount) = 0.77 ≠ 0.99 (approuvé↔versé) et corrélations à la cible de signes opposés (+0.10 vs −0.29)

**Catégorielles gardées (6)** — validées par le dépistage
- `suspicious_documents` (YES : 93% de fraude, capte 38% des fraudes), `gps_location_match` (NO : 49%), `delayed_reporting` (YES : 38.5%), `claim_submitted_at_night` (YES : 34%), `accident_severity` (TOTAL LOSS : 28%), `police_report` (NO : 24%)

**Clés & dates (métadonnées, pas des features)**
- `claim_id` : traçabilité + contrôle de groupe pour le découpage train/test
- `claim_date`, `incident_date` : découpage temporel + calcul du délai de déclaration

**Exclusions (et pourquoi)**
- `fraud_risk_score` : score produit par le moteur de risque qui définit la cible (corr 0.81)
- `approved_amount`, `payout_amount` : montants approuvé/versé par l'assureur, inconnus au moment du dépôt
- `claim_duration_days` : définition non documentée, durée de traitement probable (post-décision) — remplacé par le délai de déclaration recalculé
- `claim_status` : 100% des fraudes sont en DENIED → proxy de la cible
- `identity_verified`, `inconsistent_statements`, `repair_shop_verified`, `claim_history_risk` : résultats d'une vérification ou d'une enquête, indisponibles au dépôt
- `total_loss` : doublon exact d'`accident_severity` (TOTAL LOSS)
- `incident_type`, `multiple_claims_last_year` + les 16 catégorielles plates : aucun signal (taux ~20%)
- `vehicle_year` : doublon de `vehicle_age` (corr −0.97)
- `customer_name` (PII), `incident_location`, `vehicle_make`, `vehicle_model` : cardinalité élevée sans signal
- `_rescued_data` + colonnes d'audit (`source_bronze_table`, `silver_batch_id`, `silver_ingestion_timestamp`) : techniques

**Features créées (feature engineering)** — justifiées par la doc ("Target Generation Logic")
- `reporting_delay_days` = `claim_date − incident_date` (facteur documenté "delayed incident reporting")
- `repair_cost_ratio` = `repair_cost / vehicle_value` (coût de réparation surfacturé par rapport à la valeur du véhicule)
- `claim_amount_ratio` = `claim_amount / vehicle_value`
- `severe_no_police_report` = 1 si `accident_severity` ∈ (MAJOR, TOTAL LOSS) ET `police_report` = NO (facteur documenté "severe accidents without police reports")

#### Création de la table Gold

- encodage de la cible (fraudulent) : 
    - YES -> 1
    - NO -> 0
    - Type : INT

In [0]:
# Paramètres des tables
silver_table_name = "workspace.projet_alpha.insurance_claim_fraud_silver"
gold_table_name = "workspace.projet_alpha.insurance_claim_fraud_gold"

In [0]:
# Construction de la Gold : features sélectionnées + cible encodée + features créées
gold_table = spark.sql(f"""SELECT claim_id,
                       claim_date,
                       incident_date,
                       age,
                       credit_score,
                       years_with_company,
                       previous_claims,
                       annual_income,
                       policy_duration_years,
                       premium_amount,
                       coverage_amount,
                       deductible,
                       witnesses,
                       injuries,
                       vehicle_age,
                       mileage,
                       vehicle_value,
                       repair_cost,
                       towing_cost,
                       medical_cost,
                       rental_vehicle_cost,
                       claim_amount,
                       accident_severity,
                       CASE WHEN fraudulent = 'YES' THEN 1 ELSE 0 END AS is_fraud,
                       CASE WHEN suspicious_documents = 'YES' THEN 1 ELSE 0 END AS is_suspicious,
                       CASE WHEN gps_location_match = 'YES' THEN 1 ELSE 0 END AS is_gps_location_match,
                       CASE WHEN delayed_reporting = 'YES' THEN 1 ELSE 0 END AS is_delayed_reported,
                       CASE WHEN claim_submitted_at_night = 'YES' THEN 1 ELSE 0 END AS is_night_claim,
                       CASE WHEN police_report = 'YES' THEN 1 ELSE 0 END AS has_police_report,
                       DATEDIFF(claim_date, incident_date) AS reporting_delay_days,
                       ROUND(repair_cost / NULLIF(vehicle_value, 0), 2) AS repair_cost_ratio,
                       ROUND(claim_amount / NULLIF(vehicle_value, 0), 2) AS claim_amount_ratio,
                       CASE WHEN accident_severity IN ('MAJOR', 'TOTAL LOSS') AND police_report = 'NO' THEN 1 ELSE 0 END AS severe_no_police_report
                       FROM {silver_table_name}""")
display(gold_table)

In [0]:
# Sauvegarder la table Gold en table Delta
gold_table.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_table_name)

print(f"Table Gold sauvegardée dans {gold_table_name}")

#### Vérification de la table Gold

- Volume : 100 000 lignes (rien filtré)
- Schéma : le compte de colonnes — exactement la liste prévue (19 num + 6 cat + cible + 3 métadonnées + 4 créées = 33)
- Cible intacte : taux de fraude toujours 20,04 / 79,96 — si ça bouge, la cible a été transformée par erreur

In [0]:
# Contrôles finaux : nb de lignes, colonnes, taux de fraude
nb_count = gold_table.count()
print(f"\nNombre de lignes dans la table Gold: {nb_count}")

colonnes_names_gold = [f.name for f in gold_table.schema.fields]
print(f"\nColonnes dans la table Gold:\n{colonnes_names_gold}")
print(f"\nNombre de colonnes dans la table Gold: {len(colonnes_names_gold)}\n")

taux_fraude_gold = spark.sql(f"""
    SELECT count(CASE WHEN is_fraud = 1 THEN 1 END) AS nb_fraudes,
    count(CASE WHEN is_fraud = 0 THEN 1 END) AS nb_non_fraudes,
    count(*) AS nb_total,
    round(count(CASE WHEN is_fraud = 1 THEN 1 END) / count(*) * 100, 2) AS taux_fraude_pct,
    round(count(CASE WHEN is_fraud = 0 THEN 1 END) / count(*) * 100, 2) AS taux_non_fraude_pct
    FROM {gold_table_name}
""")
display(taux_fraude_gold)